# 02 · LLM, concordância e catálogo de erros
Etapas 03.2–03.4. **Ordem obrigatória:** rótulos A e B commitados antes de rodar o LLM.

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd, matplotlib.pyplot as plt
from src import config, limpeza
plt.rcParams.update({'figure.figsize': (8, 3.5), 'axes.spines.top': False, 'axes.spines.right': False})
from src import rotulagem, concordancia as c
posts = pd.read_parquet(config.POSTS_PARQUET)
ra = rotulagem.ler_rotulos(config.DATA_PROC/'rotulos_A.csv')
rb = rotulagem.ler_rotulos(config.DATA_PROC/'rotulos_B.csv')
rotulagem.kappa_entre_humanos(ra, rb)

{'n': 100,
 'kappa_sentimento': 0.8107255520504731,
 'kappa_sentimento_ponderado': 0.8450413223140496,
 'kappa_ironia': 0.9681528662420382,
 'concordancia_bruta_ironia': 0.99,
 'kappa_giria_meme': 0.5806451612903225,
 'concordancia_bruta_giria_meme': 0.87,
 'kappa_suspeita_coordenacao': 1.0,
 'concordancia_bruta_suspeita_coordenacao': 1.0,
 'kappa_irrelevante': 0.8120300751879699,
 'concordancia_bruta_irrelevante': 0.93}

## Kappa entre humanos e divergências

In [2]:
rotulagem.divergencias(ra, rb)[['id','sentimento_A','sentimento_B']]

,id,sentimento_A,sentimento_B
0,2,positivo,positivo
1,7,positivo,neutro
2,8,positivo,neutro
3,9,positivo,positivo
4,11,negativo,negativo
5,12,neutro,neutro
6,14,neutro,positivo
7,17,positivo,positivo
8,19,neutro,neutro
9,21,neutro,neutro


## Kappa humano × LLM (v1 e v2)
Gabarito = `rotulos_consenso.csv`. Rodar antes: `llm --versao v1 --avaliacao` e `--versao v2 --avaliacao`.

In [3]:
gab = rotulagem.ler_rotulos(config.DATA_PROC/'rotulos_consenso.csv')
res = {}
for v in ('v1','v2'):
    llm = pd.read_parquet(config.DATA_PROC/f'sentimento_llm_aval_{v}.parquet')
    res[v] = c.juntar(gab, llm, posts)
c.tabela_kappas(res).round(3)

,versao,n,kappa,kappa_ponderado,acuracia,f1_macro,n_sem_ironia,kappa_sem_ironia,acuracia_sem_ironia,n_com_ironia,kappa_com_ironia,acuracia_com_ironia
0,v1,100,0.593,0.634,0.73,0.732,80,0.636,0.762,20,0.316,0.6
1,v2,100,0.615,0.620,0.75,0.749,80,0.630,0.762,20,0.417,0.7


## Matrizes de confusão

In [4]:
for v, m in res.items():
    print(v); print(c.matriz_confusao(m)); print()

v1
              llm_negativo  llm_neutro  llm_positivo
hum_negativo            19           0             1
hum_neutro               9          27             8
hum_positivo             3           6            27

v2
              llm_negativo  llm_neutro  llm_positivo
hum_negativo            16           1             3
hum_neutro               4          30            10
hum_positivo             3           4            29



## Detecção de ironia pelo LLM

In [5]:
{v: c.deteccao_ironia(m) for v, m in res.items()}

{'v1': {'vp': 14,
  'fp': 3,
  'fn': 6,
  'precisao': 0.8235294117647058,
  'recall': 0.7},
 'v2': {'vp': 12,
  'fp': 2,
  'fn': 8,
  'precisao': 0.8571428571428571,
  'recall': 0.6}}

## Catálogo de padrões de falha
1. Exporte `discordancias_v*.csv` (já gerado por `run_pipeline concordancia`).
2. **Preencha manualmente** `categoria_erro_revisada` e `comentario`.
3. Rode a célula abaixo. Para cada categoria, escreva no relatório o *porquê* (o que o LLM não sabia) com **exemplos inventados/parafraseados**, nunca o texto original.

In [6]:
d = pd.read_csv(config.DATA_PROC/'discordancias_v2.csv')
cat = c.catalogo_erros(d.fillna(''))
cat

,categoria,n,conf_media_llm,pct,n_conf_alta
0,outro,12,0.758333,0.48,6
1,ironia,6,0.833333,0.24,6
2,giria_meme,3,0.766667,0.12,1
3,ticker_ambiguo,3,0.866667,0.12,3
4,manipulacao,1,0.800000,0.04,1


## Erros de alta confiança
"Justificativa convincente não é acerto": quantos erros o LLM cometeu com confiança ≥ 0,8?

In [7]:
d[d['confianca']>=0.8][['id','sentimento_hum','sentimento_llm','confianca','categoria_erro']]

,id,sentimento_hum,sentimento_llm,confianca,categoria_erro
1,8,neutro,negativo,0.8,ticker_ambiguo
2,10,positivo,negativo,0.8,ironia
3,12,neutro,positivo,0.8,ironia
4,15,neutro,negativo,0.9,ironia
5,19,neutro,positivo,0.9,giria_meme
6,33,negativo,positivo,0.8,outro
9,45,neutro,positivo,0.9,ironia
10,46,positivo,neutro,0.8,outro
11,48,neutro,negativo,0.8,ironia
15,64,neutro,negativo,0.8,outro


## Custo
Ver `data/processed/custo_llm.jsonl`.

In [8]:
import json
pd.DataFrame([json.loads(l) for l in open(config.DATA_PROC/'custo_llm.jsonl')])[['versao','modelo','total','em_cache','tokens_in','tokens_out','custo_usd']]

,versao,modelo,total,em_cache,tokens_in,tokens_out,custo_usd
0,v1,gemini-3.1-flash-lite,100,0,3196,2788,0.001076
1,v2,gemini-3.1-flash-lite,100,0,6511,2792,0.001326
2,v1,gemini-3.1-flash-lite,100,100,0,0,0.000000
3,v2,gemini-3.1-flash-lite,100,100,0,0,0.000000
